### TASK 1

### 1.2 Import Libraries

In [ ]:
# import all the libraries i need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score

import warnings
warnings.filterwarnings('ignore')

print('libraries loaded')

Libraries loaded successfully.


### 1.3 Load Dataset

In [ ]:
# load the dataset
df = pd.read_excel('Online Retail.xlsx', engine='openpyxl')

# check the shape
print('shape of dataset:', df.shape)

# look at first few rows
df.head()
# check column names and types
print(df.columns)
print()
print(df.dtypes)
# check for missing values
print('missing values in each column:')
print(df.isnull().sum())

Raw dataset shape: (541909, 8)
Column names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate    float64
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,40513.351389,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,40513.351389,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,40513.351389,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,40513.351389,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,40513.351389,3.39,17850.0,United Kingdom


### 1.4 Data Pre-processing

In [ ]:
# step 1 - drop rows where CustomerID is missing
# we need CustomerID to group transactions by customer
df = df.dropna(subset=['CustomerID'])
print('rows after dropping missing CustomerID:', len(df))

Missing values before cleaning:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Rows after dropping missing CustomerID: 406,829


In [ ]:
# step 2 - remove cancelled orders
# cancelled orders have invoice numbers starting with C
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print('rows after removing cancellations:', len(df))

# step 3 - remove rows with negative or zero quantity/price
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
print('rows after removing bad quantity/price:', len(df))

# step 4 - convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# step 5 - create TotalPrice column
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# also convert CustomerID to int
df['CustomerID'] = df['CustomerID'].astype(int)

print('done')
df.head()

Rows after removing cancellations: 397,924
Rows after removing invalid Quantity/UnitPrice: 397,884

Cleaned dataset sample:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,1970-01-01 00:00:00.000040513,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,1970-01-01 00:00:00.000040513,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,1970-01-01 00:00:00.000040513,2.75,17850,United Kingdom,22.00


In [ ]:
# step 6 - build the RFM table
# snapshot date is the day after the last transaction
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print('snapshot date:', snapshot_date)

# group by customer and calculate R, F, M
rfm = df.groupby('CustomerID').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary = ('TotalPrice', 'sum')
).reset_index()

print('rfm table shape:', rfm.shape)
rfm.head()

# look at basic stats of rfm
rfm.describe()

Snapshot date (reference for Recency): 1970-01-02
RFM table shape: (4338, 4)

RFM summary statistics:
       Recency  Frequency   Monetary
count   4338.0    4338.00    4338.00
mean       1.0       4.27    2054.27
std        0.0       7.70    8989.23
min        1.0       1.00       3.75
25%        1.0       1.00     307.41
50%        1.0       2.00     674.48
75%        1.0       5.00    1661.74
max        1.0     209.00  280206.02


In [ ]:
# step 7 - remove outliers by capping at 99th percentile
# this is called winsorization
for col in ['Recency', 'Frequency', 'Monetary']:
    cap_value = rfm[col].quantile(0.99)
    rfm[col] = rfm[col].clip(upper=cap_value)
    print(col, '- capped at:', round(cap_value, 2))

    # step 8 - scale the data
# clustering uses distance so we need to scale
# otherwise monetary will dominate because its much larger
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print('scaling done')
print('shape:', rfm_scaled.shape)

Recency capped at 99th percentile: 1.00
Frequency capped at 99th percentile: 30.00
Monetary capped at 99th percentile: 19881.00

Standardisation complete.
Mean per feature (should be ~0): [ 0. -0.  0.]
Std per feature  (should be ~1): [0. 1. 1.]


In [ ]:
# Column 1: InvoiceNo
# I used value counts to check how many unique invoices there are
print('number of unique invoices:', df['InvoiceNo'].nunique())
print('sample values:')
print(df['InvoiceNo'].head())

# observation: there are many unique invoices which makes sense
# for a transactional retail dataset


# Column 2: StockCode
# top 10 most sold products by quantity
top_stock = df.groupby('StockCode')['Quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(9, 4))
top_stock.plot(kind='bar', color='steelblue')
plt.title('Top 10 Stock Codes by Total Quantity Sold')
plt.xlabel('Stock Code')
plt.ylabel('Total Quantity')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('fig_stockcode.png')
plt.show()

# observation: a small number of products make up most of the sales



# Column 3: Description
# top 15 products by total revenue
top_products = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
top_products.plot(kind='barh', color='teal')
plt.gca().invert_yaxis()
plt.title('Top 15 Products by Total Revenue')
plt.xlabel('Total Revenue (GBP)')
plt.ylabel('Product')
plt.tight_layout()
plt.savefig('fig_top_products.png')
plt.show()

# observation: a few products generate most of the revenue
# this is consistent with the 80/20 rule


# Column 4: Quantity
# use a histogram to see the distribution
# cap at 99th percentile so the graph is readable
q_cap = df['Quantity'].quantile(0.99)

plt.figure(figsize=(9, 4))
df[df['Quantity'] <= q_cap]['Quantity'].hist(bins=50, color='coral')
plt.title('Distribution of Quantity per Line Item')
plt.xlabel('Quantity')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('fig_quantity.png')
plt.show()

print('mean quantity:', round(df['Quantity'].mean(), 2))
print('median quantity:', df['Quantity'].median())

# observation: most orders are small quantities, the distribution is right skewed